# Prueba de funcionlidad `procesarOT.py`

### Cargar Librerias

In [2]:
import os
import sys
import time
from datetime import datetime
import pymongo
from pymongo.errors import ConnectionFailure
import logging

logging.basicConfig(level=logging.INFO)

from eerssa.secret import Keys

# --- MongoDB Connection ---
# It's better to establish the connection once and keep it open for the app's lifetime.
# We will also exit if the connection fails, as the consumer can't do its job without it.
uri = Keys.MONGO_KEY.value
client = None  # Initialize client to None
db_eerssa = None
CurrentCollection = None
ReloadCollection = None


try:
    # Add a timeout to avoid blocking indefinitely
    client = pymongo.MongoClient(uri, serverSelectionTimeoutMS=5000)
    # The ping command is cheap and does not require auth.
    client.admin.command('ping')
    db_eerssa = client.eerssa                   # Base de datos EERSSA
    CurrentCollection = db_eerssa.ot_v30        # Coleccion actual
    logging.info(":::: Conexion exitosa con MongoDB ::::")
    
except ConnectionFailure as e:
    logging.error(f"\n\n ><><> Error de conexion a MongoDB: {e}")
    sys.exit(1) # Exit the script if we can't connect to MongoDB, as it's a critical dependency.


INFO:root::::: Conexion exitosa con MongoDB ::::


In [3]:
# DASK

from dask.distributed import LocalCluster, as_completed
dask = LocalCluster().get_client()
visor_dask = dask.dashboard_link

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45741 instead
  warnings.warn(
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:41869
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:45741/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:46881'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:36535'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:35851'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:45773'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:3386

In [1]:
## RECARGAR LAS LIBRERIAS DINAMICAMENTE
from importlib import reload
from eerssa import procesarOt as OrdenTrabajo             # Convert from PDF_ot to obj_ot
from eerssa import generarMatrizActividades as Actividades     # process ot.data["actividades"]

Success!!!


### Recargar Librerias

In [24]:
reload( OrdenTrabajo )
reload( Actividades  )
#print(visor_dask)

<module 'eerssa.generarMatrizActividades' from '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/eerssa/generarMatrizActividades.py'>

### Directorios de Prueba

In [2]:
from pathlib import Path

dir_test = Path ("/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test")

# Get all entries (files and subdirectories)
all_entries = dir_test.iterdir()

# Filter for only files
files = [item for item in all_entries if item.is_file()]

# You can also get just the names if you prefer
# file_names = [item.name for item in all_entries if item.is_file()]

print("All files in the directory (Path objects):")
print(files)

# If you need them as strings
file_strings = [str(f) for f in files]
print("\nAll files as strings:")
print(file_strings)

All files in the directory (Path objects):
[PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/28_6 colaboradores.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/7_one_line_text_overlap.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/11_LM_Tres_hojas.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/10_Test Orden de trabajo Zamora 11-02-2022 (RM - Electricistas).pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/12_casoEspecial01.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/14_Orden de trabajo Guayzimi 10-04-2022 (CQ).pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/27_5 colaboradores.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/8_bad_line_text_overlap.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/21_0 colaboradores y vehiculo.pdf'), PosixPath('/

### Test con una sola hoja

In [3]:
nro_ot_test =20
ot_test = OrdenTrabajo.procesarOt(file_strings[ nro_ot_test ])
ot_test

{'version': '0.3.0',
 'link': '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/30_Repetido_Responsable_Colaborador.pdf',
 'exito': True,
 'log': [{'t': '2025-08-08T23:43:17.776337',
   'level': 'INFO',
   'message': 'CREACION DE LA OT, se encuentra un archivo PDF valido',
   'detail': 'Ubicacion: /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/30_Repetido_Responsable_Colaborador.pdf'},
  {'t': '2025-08-08T23:43:18.005721',
   'level': 'REVISAR',
   'message': 'Se reportan CARENCIAS',
   'detail': 'Revisar si estan reportadas CARENCIAS'}],
 'createdAt': '2025-08-08T23:43:17.785047-05:00',
 'terminado': 'TERMINADO',
 'id_ot': 103411,
 'cuadrilla': 'Loja Z1 Linieros Nocturnos',
 'responsable': ['PUZMA ALDAZ RENE MICHAEL', 'TELE'],
 'colaboradores': {'total': 0, 'nombres': []},
 'diaSemana': 'miércoles',
 'fecha': '2023-03-01T00:00:00-05:00',
 'fechaString': 'miércoles, 01 de marzo del 2023',
 'tEstimado': '10 HORAS',
 'fechaFinal': '02/03/2023 08:00:00',
 'vehiculo': '

### DASK Parallel Computing

#### BAGS

In [7]:
import dask.bag as db
import dask
from datetime import datetime

file_strings

print(f"Se han encontrado un total de: {len(file_strings)} Ordenes de Trabajo")

start_time = time.time()
start_datetime = datetime.now()
print(f"Hora de inicio: {start_datetime.strftime('%Y-%m-%d %H:%M:%S')}\n\n")

# 1. Create a Dask Bag from the list of PDF file paths
# Dask Bags are great for unstructured or custom data processing
dask_bag = db.from_sequence(file_strings)

# 2. Use the .map() method to apply the procesarOt function to each item
# This creates a computation graph. The function is not executed yet.
# `procesarOt` should be a standalone function, as it is in your provided code.
mapped_bag = dask_bag.map(OrdenTrabajo.procesarOt)

# 3. Call .compute() to trigger the parallel execution and get the final list of results.
# Dask will handle the scheduling of these tasks across workers.
obj_lists_dask = mapped_bag.compute()

end_time = time.time()
elapsed_time = end_time - start_time
end_datetime = datetime.now()
print(f"\n\n   Procesados todos los {len(obj_lists_dask)} items. Tiempo transcurrido: {elapsed_time:.2f} segundos.\n   Hora Final : {end_datetime.strftime('%Y-%m-%d %H:%M:%S')}")

Se han encontrado un total de: 28 Ordenes de Trabajo
Hora de inicio: 2025-08-03 15:20:53


Success!!!
Success!!!
Success!!!
Success!!!


   Procesados todos los 28 items. Tiempo transcurrido: 2.22 segundos.
   Hora Final : 2025-08-03 15:20:55


#### Futures

In [7]:
# Verificar la conversion de archivos

print(f" Se han encontrado un total de: {len(file_strings)} Ordenes de Trabajo" )

start_time = time.time()
start_datetime = datetime.now()
print( f"Hora de inicio: {start_datetime.strftime('%Y-%m-%d %H:%M:%S')}\n\n" )

# 1. Submit the first batch of tasks
# This returns a list of futures, same as before.
futures_step1 = [dask.submit(OrdenTrabajo.procesarOt, file, pure=False) for file in file_strings]

# 2. Submit the second batch of tasks, feeding the first futures as input
#futures_step2 = [dask.submit(call_load_ot, f) for f in futures_step1]

# 3. Now, gather only the FINAL results
# This single call executes the entire graph (both GestionOt and load_ot) in parallel.
obj_lists_dask = dask.gather(futures_step1)


end_time = time.time()
elapsed_time = end_time - start_time
end_datetime = datetime.now()
print(f"\n\n   Procesados todos los {len(obj_lists_dask)} items. Tiempo transcurrido: {elapsed_time:.2f} segundos.\n   Hora Final : {end_datetime.strftime('%Y-%m-%d %H:%M:%S')}")



 Se han encontrado un total de: 28 Ordenes de Trabajo
Hora de inicio: 2025-08-04 00:43:21


Success!!!
Success!!!
Success!!!
Success!!!
MuPDF error: format error: cannot recognize version marker



   Procesados todos los 28 items. Tiempo transcurrido: 3.16 segundos.
   Hora Final : 2025-08-04 00:43:24


In [78]:
dask.cancel(futures_step1)

In [ ]:
dask.close()

In [13]:
obj_lists_dask[27]

NameError: name 'obj_lists_dask' is not defined

# DIBUJAR OTS

In [6]:
# Import your existing modules
from importlib import reload
import os
from eerssa.constants import BoxesValues as boxes
import pymupdf
from   pymupdf import Rect



path_page = '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/3_ok_cl_21_items.pdf'

In [28]:
itemX1 = 28
itemY1 = 92

itemX2 = 41
itemY2 = 117.85

deltaY = 28.32

deltaXactiv1 = 15
anchoActiv = 36

deltaEvento = 53
anchoEvento = 298

deltaLMT = 340
anchoLMT = 397

deltaTipo = 440
anchoTipo = 490

deltaInicio = 491
anchoFecha = 37

deltaFin = 529.5

kilomY = 58
KilomD = 9
kmInicioX1 = 140
kmInicioX2 = 55

kmFinalX1 = 267
kmFinalX2 = 55

kmRecorX1 = 417
kmRecorX2 = 55

vehicX1 = 125
placaX1 = 190
rentadoX1 = 286
choferX1 = 342
vehicDl = 27

vehicY1 = 46.5 
vehiYdl = 9

firmasX1 = 85
firmasAncho = 159
firmasDelta = 160

firmasY1 = 740
firmasAlto = 23.4

fechaInicioY1 = 263
fechaDeltaY   = 11

fechaInicioX1 = 100
fechaAncho    = 170

duracionX1    = 483
duracionAncho = 80



with pymupdf.open( path_page ) as pdf:
  
  hoja1 = pdf[0].new_shape()

  #Fecha mitad de hoja
  hoja1.draw_rect( Rect( fechaInicioX1, fechaInicioY1, fechaInicioX1+fechaAncho, fechaInicioY1+ fechaDeltaY ))
  
  # Duración tEstimado
  hoja1.draw_rect( Rect( duracionX1, fechaInicioY1, duracionX1+duracionAncho, fechaInicioY1+ fechaDeltaY) )

  for i in range(3):
    hoja1.draw_rect(Rect( firmasX1 + (i*firmasDelta),
                          firmasY1,
                          firmasX1 + (i*firmasDelta) + firmasAncho,
                          firmasY1+firmasAlto     ))

  hoja1.finish( color=(0, 1, 0), fill=None )
  hoja1.commit()


# --------------------------------


  hoja2 = pdf[1].new_shape()

  # Datos de numero
  hoja2.draw_rect(Rect( vehicX1, vehicY1 , vehicX1 + vehicDl*1.2, vehicY1+vehiYdl ))

  # Placa
  hoja2.draw_rect(Rect( placaX1, vehicY1 , placaX1 + vehicDl*2, vehicY1+vehiYdl ))

  # Rentado
  hoja2.draw_rect(Rect( rentadoX1, vehicY1 , rentadoX1 + vehicDl*0.8, vehicY1+vehiYdl ))

  # Chofer
  hoja2.draw_rect(Rect( choferX1, vehicY1 , choferX1 + vehicDl*8, vehicY1+vehiYdl ))


  # Kilometraje Inicial
  hoja2.draw_rect(Rect( kmInicioX1, kilomY , kmInicioX1 + kmInicioX2,kilomY+KilomD  ))

  #Kilometraje Final
  hoja2.draw_rect(Rect( kmFinalX1, kilomY , kmFinalX1 + kmFinalX2,kilomY+KilomD  ))

  #Kilometraje Total
  hoja2.draw_rect(Rect( kmRecorX1, kilomY , kmRecorX1 + kmRecorX2,kilomY+KilomD  ))

  # Actividades
  for i in range(21):
    # ITEM
    hoja2.draw_rect( 
      Rect(itemX1, itemY1+(i*deltaY), itemX2, itemY2+(i*deltaY)))
    
    # Activ
    hoja2.draw_rect( 
      Rect(
        itemX1 + deltaXactiv1, 
        itemY1+(i*deltaY), 
        itemX2 + anchoActiv, 
        itemY2+(i*deltaY) ))
    
    #Evento
    hoja2.draw_rect( 
      Rect(
        itemX1 + deltaEvento, 
        itemY1+(i*deltaY), 
        itemX2 + anchoEvento, 
        itemY2+(i*deltaY) ))
    
    #Alimentador
    hoja2.draw_rect( 
      Rect(
        itemX1 + deltaLMT, 
        itemY1+(i*deltaY), 
        itemX2 + anchoLMT, 
        itemY2+(i*deltaY) ))
    
    # Tipo
    hoja2.draw_rect( 
      Rect(
        deltaTipo, 
        itemY1+(i*deltaY), 
        anchoTipo, 
        itemY2+(i*deltaY) ))
    
    # Inicio
    hoja2.draw_rect( 
      Rect(
        deltaInicio, 
        itemY1+(i*deltaY), 
        deltaInicio+anchoFecha, 
        itemY2+(i*deltaY) ))
    
    # Fin
    hoja2.draw_rect( 
      Rect(
        deltaFin, 
        itemY1+(i*deltaY), 
        deltaFin+anchoFecha, 
        itemY2+(i*deltaY) ))

  hoja2.finish( color=(0, 1, 0), fill=None )
  hoja2.commit()

  pdf.save("Matriz_hoja2.pdf")